In [ ]:
# Naive Baselines for Intraday Price Indices
# (i)   Naive1: Previous hour price.
# (ii)  Naive2: Same delivery hour previous day (t-24h).
# (iii) Naive3: Average of same delivery hour previous 3 days (t-24h, t-48h, t-72h).
# Probabilistic extension: For each baseline we derive empirical residual quantiles (0.1,0.5,0.9)
# by hour-of-day on the training set and add them to the point prediction to form quantile forecasts.

import os
import math
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------ Configuration ------------------ #
TRAIN_START_DATE = '2022-01-01'
SPLIT_LEN = (24, 6, 6)   # (train_months, val_months, test_months) – mirrors Main.py
QUANTILES = [0.1, 0.5, 0.9]
COUNTRIES = ['germany', 'austria']
INDICES = ['ID1', 'ID2', 'ID3']
RESOLUTION = 'h'  # hourly
DATA_DIR = 'Data'

TRAIN_START_DATE = os.environ.get('NAIVE_TRAIN_START', TRAIN_START_DATE)
print(f"Config -> train_start={TRAIN_START_DATE}, split(months)={SPLIT_LEN}, quantiles={QUANTILES}")

In [ ]:
def load_labels(country: str, indice: str) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, f"Label_{RESOLUTION}_{country}_{indice}.pkl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing label file: {path}")
    df = pd.read_pickle(path).copy()
    # Expect columns: ['Date_DeliveryStart', indice, ...]
    df = df[['Date_DeliveryStart', indice]]
    df.rename(columns={'Date_DeliveryStart': 'ts', indice: 'price'}, inplace=True)
    df['ts'] = pd.to_datetime(df['ts'], utc=True)
    df.sort_values('ts', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


def derive_splits(df: pd.DataFrame):
    train_months, val_months, test_months = SPLIT_LEN
    start = pd.to_datetime(TRAIN_START_DATE, utc=True)
    train_end = start + pd.DateOffset(months=train_months)
    val_end = train_end + pd.DateOffset(months=val_months)
    test_end = val_end + pd.DateOffset(months=test_months)
    train = df[(df.ts >= start) & (df.ts < train_end)]
    val = df[(df.ts >= train_end) & (df.ts < val_end)]
    test = df[(df.ts >= val_end) & (df.ts < test_end)]
    return train, val, test


def compute_naive_point_predictions(df: pd.DataFrame):
    # Assumes hourly regular spacing; we'll rely on timestamp alignment.
    df = df.copy()
    df['naive1'] = df['price'].shift(1)  # previous hour
    df['naive2'] = df['price'].shift(24)  # same hour previous day
    # average same hour previous 3 days: hours -24, -48, -72
    shifts = [24, 48, 72]
    df['naive3'] = df[shifts and 'price']  # temporary line to avoid linter
    # Create columns for each shift then average
    for s in shifts:
        df[f'lag_{s}'] = df['price'].shift(s)
    df['naive3'] = df[[f'lag_{s}' for s in shifts]].mean(axis=1)
    return df


def build_quantile_adjustments(train_df: pd.DataFrame, baseline_col: str):
    # Residuals grouped by hour-of-day; compute empirical quantiles.
    tmp = train_df.dropna(subset=[baseline_col, 'price']).copy()
    tmp['resid'] = tmp['price'] - tmp[baseline_col]
    tmp['hour'] = tmp['ts'].dt.hour
    quant_table = tmp.groupby('hour')['resid'].quantile(QUANTILES).unstack(level=-1)
    quant_table.columns = [f"q{int(q*100)}" for q in QUANTILES]
    return quant_table  # columns q10 q50 q90


def apply_quantile_forecasts(df: pd.DataFrame, quant_table: pd.DataFrame, baseline_col: str):
    # Merge hour quantiles
    out = df.copy()
    out['hour'] = out['ts'].dt.hour
    out = out.merge(quant_table, on='hour', how='left')
    # Quantile forecasts = baseline + residual quantiles
    for q in QUANTILES:
        out[f'{baseline_col}_q{int(q*100)}'] = out[baseline_col] + out[f'q{int(q*100)}']
    # Keep minimal columns
    keep_cols = ['ts', 'price', baseline_col] + [f'{baseline_col}_q{int(q*100)}' for q in QUANTILES]
    return out[keep_cols]


def pinball_loss(y_true, y_pred, q: float):
    err = y_true - y_pred
    return np.mean(np.maximum(q*err, (q-1)*err))


def evaluate_quantile_baseline(df: pd.DataFrame, baseline_col: str):
    # df contains columns: price, baseline_col, baseline_col_q10/_q50/_q90
    records = {}
    y_true = df['price'].values
    median_pred = df[f'{baseline_col}_q50'].values
    # Deterministic metrics from median
    rmse = math.sqrt(mean_squared_error(y_true, median_pred))
    mae = mean_absolute_error(y_true, median_pred)
    r2 = r2_score(y_true, median_pred)
    q_losses = []
    for q in QUANTILES:
        q_pred = df[f'{baseline_col}_q{int(q*100)}'].values
        ql = pinball_loss(y_true, q_pred, q)
        q_losses.append(ql)
        records[f'Q{q}'] = ql
    records['AQL'] = float(np.mean(q_losses))
    records['RMSE'] = rmse
    records['MAE'] = mae
    records['R2'] = r2
    return records


# ------------------ Main Execution ------------------ #
all_results = []
for country in COUNTRIES:
    for indice in INDICES:
        try:
            df_raw = load_labels(country, indice)
        except FileNotFoundError as e:
            print(e)
            continue
        df_full = compute_naive_point_predictions(df_raw)
        train_df, val_df, test_df = derive_splits(df_full)
        # We'll compute residual quantiles using ONLY train set for each baseline
        for baseline in ['naive1', 'naive2', 'naive3']:
            if baseline not in train_df.columns:  # safety
                continue
            quant_table = build_quantile_adjustments(train_df, baseline)
            test_with_q = apply_quantile_forecasts(test_df, quant_table, baseline)
            # Drop rows with missing baseline (early period)
            test_eval = test_with_q.dropna()
            if len(test_eval) == 0:
                print(f"No evaluable rows for {country}-{indice}-{baseline}")
                continue
            metrics = evaluate_quantile_baseline(test_eval, baseline)
            metrics.update({'Country': country, 'Indice': indice, 'Baseline': baseline})
            all_results.append(metrics)

results_df = pd.DataFrame(all_results)
# Reorder columns
cols_order = ['Country', 'Indice', 'Baseline', 'Q0.1', 'Q0.5', 'Q0.9', 'AQL', 'RMSE', 'MAE', 'R2']
results_df = results_df[cols_order]
print("Naive baselines results (test set):")

# Expect 18 rows if all files present
print(f"Rows produced: {len(results_df)} (expected 18)")

results_df

Naive baselines results (test set):
Rows produced: 18 (expected 18)


,Country,Indice,Baseline,Q0.1,Q0.5,Q0.9,AQL,RMSE,MAE,R2
0,germany,ID1,naive1,5.703298,8.998069,5.644584,6.781983,45.973287,17.996138,0.741551
1,germany,ID1,naive2,13.264381,21.276823,13.223555,15.921586,99.527024,42.553647,-0.393290
2,germany,ID1,naive3,12.804201,21.043507,13.669071,15.838926,94.624444,42.087014,-0.095355
3,germany,ID2,naive1,4.354252,7.012731,4.327855,5.231613,35.036834,14.025462,0.817786
4,germany,ID2,naive2,11.990453,19.636964,12.321672,14.649696,89.581403,39.273929,-0.191228
5,germany,ID2,naive3,11.904723,19.364771,12.252292,14.507262,82.922746,38.729542,-0.020843
6,germany,ID3,naive1,3.913627,6.388030,3.869569,4.723742,28.491185,12.776060,0.866807
7,germany,ID3,naive2,11.443894,18.961552,11.831551,14.078999,81.705283,37.923105,-0.095293
8,germany,ID3,naive3,11.614113,18.869342,11.816339,14.099931,77.573853,37.738684,0.012665
9,austria,ID1,naive1,7.593029,12.971621,7.540288,9.368313,61.589629,25.943243,0.442468


### Notes
- Quantile adjustments are derived from training residual distributions per hour-of-day.
- Validation split is unused here (could be used for calibration refinement if desired).
- If some early timestamps lack enough lag history, those rows are automatically dropped for evaluation.
- All metrics are computed on the test window only.
